# Controller T3 real CUDA engineering gates

Private dataset + T4 GPU + Internet. Frozen Qwen/NF4, CPU FP32 reranker, prefill256. No QA gold, selection or final inputs.
Eight synthetic long-input paths (>=14k token), true CPU scoring timing and short Full checks; no benchmark inference.

In [ ]:
import os, sys, json, hashlib, shutil, subprocess, tarfile, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable CUDA GPU and Internet'
WORK=Path('/kaggle/working')
BUNDLE=Path('/tmp')/('controller_'+'4fc66d4194ac')/'controller_t3_gpu_v1'
OUTPUT=WORK/'controller_t3_engineering_v1_outputs'
assert not OUTPUT.exists(), 'Preserve previous outputs'
def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda:f.read(8*1024*1024),b''):h.update(part)
    return h.hexdigest()
if not BUNDLE.exists():
    archives=list(Path('/kaggle/input').rglob('controller_t3_gpu_v1.tar.gz'))
    indexes=[p for p in Path('/kaggle/input').rglob('bundle_checksums.json') if file_sha(p)=='4fc66d4194ac947682c072e642f349881f9b70c60d89b87d7b8a54ee68cd62f4']
    BUNDLE.parent.mkdir(parents=True,exist_ok=True)
    if indexes:
        assert len(indexes)==1
        shutil.copytree(indexes[0].parent,BUNDLE)
    else:
        assert len(archives)==1 and file_sha(archives[0])=='2eccb0d8d141442b79d25e0a160c6e4bf561ec333165d18c5701971ea9a2fbf2', 'Attach the private code dataset'
        with tarfile.open(archives[0]) as t:t.extractall(BUNDLE.parent,filter='data')
index=BUNDLE/'bundle_checksums.json'
assert file_sha(index)=='4fc66d4194ac947682c072e642f349881f9b70c60d89b87d7b8a54ee68cd62f4'
for rel,digest in json.loads(index.read_text()).items():
    assert not Path(rel).is_absolute() and '..' not in Path(rel).parts
    assert file_sha(BUNDLE/rel)==digest,rel
print('Frozen code and development inputs verified; no gold/selection/final input files')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r',str(BUNDLE/'requirements-controller-t3.txt')],check=True)
subprocess.run([sys.executable,'-m','unittest','tests.test_controller_t3_execution','tests.test_controller_t3_evaluation',
                'tests.test_controller_integration','tests.test_controller_evaluation','tests.test_controller_t3_gate','-q'],cwd=BUNDLE,check=True)
subprocess.run([sys.executable,'scripts/run_controller_t4.py','--preflight','--output-dir',str(OUTPUT)],cwd=BUNDLE,check=True)


In [ ]:
command=[sys.executable,'-u','scripts/run_controller_t4.py','--output-dir',str(OUTPUT)]
command+=['--engineering-only']
log=WORK/'controller_t3_t4_run.log'
with log.open('x') as handle:
    p=subprocess.Popen(command,cwd=BUNDLE,env={**os.environ,'PYTHONUNBUFFERED':'1'},stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        print(line,end='',flush=True);handle.write(line);handle.flush()
    returncode=p.wait()
OUTPUT.mkdir(exist_ok=True)
shutil.copyfile(log,OUTPUT/'run.log')
shutil.copyfile(BUNDLE/'execution_manifest.json',OUTPUT/'input_execution_manifest.json')
print('Process return code:',returncode,'; exporting all partial or completed outputs')


In [ ]:
assert OUTPUT.exists()
checks={str(p.relative_to(OUTPUT)):file_sha(p) for p in OUTPUT.rglob('*') if p.is_file() and p.name!='output_checksums.json'}
(OUTPUT/'output_checksums.json').write_text(json.dumps(checks,indent=2))
archive=WORK/(OUTPUT.name+'.zip')
with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED,strict_timestamps=False) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file():z.write(p,str(p.relative_to(OUTPUT)))
print('Exported:',archive,'SHA256:',file_sha(archive))
assert returncode==0, 'Preserve failed run and partial outputs; no semantic retry'
